## How Do Computers Solve Math?

When you type `(100 + 200) * 3` into a calculator, you instinctively know to add first, then multiply. But **computers don't think like humans** — they need a systematic algorithm to handle:

- **Operator precedence** (`*` before `+`)
- **Parentheses** (grouping overrides precedence)
- **Left-to-right evaluation**

The solution? **Reverse Polish Notation (RPN)** — a technique invented by Polish mathematician Jan Łukasiewicz and popularized by Hewlett-Packard calculators in the 1970s.

### The Pipeline

```
Expression String → Tokenize → Convert to RPN → Evaluate → Result
"(100 + 200) * 3" → [100, +, 200, *, 3] → [100, 200, +, 3, *] → 900
```

### Key Data Structures

| Structure | Used For | Why |
|-----------|----------|-----|
| **ArrayList** | Storing tokens & RPN output | Ordered sequence, index access |
| **Stack** | Operator reordering & calculation | LIFO — last operator in is first out |
| **HashMap** | Looking up operators by character | O(1) lookup for precedence & calculation |

Let's build each piece step by step.

---

## Step 1: The Token Class

Every part of a math expression is a **Token** — a number, an operator (`+`, `-`, `*`, `/`), or a parenthesis.

The `Token` class stores:
- The **character** (e.g. `+`, `*`)
- The **precedence** (multiplication before addition)
- The **calculation** function (what the operator actually does)

**Key Concept:** We use Java's `BiFunction<Double, Double, Double>` — a functional interface that takes two doubles and returns a double. This lets us store math operations as data.

### Challenge: Read the Token class below, then modify it to add a `POWER` operator check.

Run the code to see the test output.

In [ ]:
import java.util.function.BiFunction;

public class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    // Constructor for passive/non-operator tokens
    public Token() { this('0'); }

    // Constructor for simple token (parenthesis)
    public Token(Character token) { this(token, 0, null, 0); }

    // Full constructor for operators
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }

    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }

    // Returns true if THIS token has higher precedence (lower number = higher priority)
    public boolean isPrecedent(Token other) {
        return this.precedence > other.getPrecedence();
    }

    // Execute the math operation
    public Double calculate(Double a, Double b) {
        return this.calculation.apply(a, b);
    }

    public String toString() { return this.token.toString(); }
}

// --- TEST ---
Token add = new Token('+', 4, (a, b) -> a + b, 2);
Token mul = new Token('*', 3, (a, b) -> a * b, 2);
Token paren = new Token('(');

System.out.println("Token '+' precedence: " + add.getPrecedence());
System.out.println("Token '*' precedence: " + mul.getPrecedence());
System.out.println("Does '+' yield to '*'? " + add.isPrecedent(mul));
System.out.println("3 + 5 = " + add.calculate(3.0, 5.0));
System.out.println("3 * 5 = " + mul.calculate(3.0, 5.0));

// TODO: Create a POWER token ('^', precedence 2) and test 2^10


{% capture challenge_token %}
Read the Token class. It represents a single element in a math expression — an operator with its precedence and calculation function. The test creates a few tokens and checks precedence. Try modifying the test to create a POWER token with precedence 2.
{% endcapture %}

{% capture code_token %}
import java.util.function.BiFunction;

public class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    // Constructor for passive/non-operator tokens
    public Token() { this('0'); }

    // Constructor for simple token (parenthesis)
    public Token(Character token) { this(token, 0, null, 0); }

    // Full constructor for operators
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }

    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }

    // Returns true if THIS token has higher precedence (lower number = higher priority)
    public boolean isPrecedent(Token other) {
        return this.precedence > other.getPrecedence();
    }

    // Execute the math operation
    public Double calculate(Double a, Double b) {
        return this.calculation.apply(a, b);
    }

    public String toString() { return this.token.toString(); }
}

// --- TEST ---
Token add = new Token('+', 4, (a, b) -> a + b, 2);
Token mul = new Token('*', 3, (a, b) -> a * b, 2);
Token paren = new Token('(');

System.out.println("Token '+' precedence: " + add.getPrecedence());
System.out.println("Token '*' precedence: " + mul.getPrecedence());
System.out.println("Does '+' yield to '*'? " + add.isPrecedent(mul));
System.out.println("3 + 5 = " + add.calculate(3.0, 5.0));
System.out.println("3 * 5 = " + mul.calculate(3.0, 5.0));

// TODO: Create a POWER token ('^', precedence 2) and test 2^10
{% endcapture %}

{% include runners/code.html
   runner_id="rpn_token"
   language="java"
   challenge=challenge_token
   code=code_token
   height="500px"
%}


<div class="rpn-popcorn">
    <div class="rpn-popcorn__header">
        <h3>Popcorn Hack 1: Precedence Check</h3>
        <p>In the Token class, lower precedence numbers mean higher priority. Given <code>Token add = new Token('+', 4, ...)</code> and <code>Token mul = new Token('*', 3, ...)</code>, what does <code>add.isPrecedent(mul)</code> return?</p>
    </div>
    <div class="rpn-popcorn__body">
        <div class="rpn-popcorn__options" id="ph1-options">
            <button onclick="checkPH1(0)" class="rpn-popcorn__option">A) true — because '+' comes first alphabetically</button>
            <button onclick="checkPH1(1)" class="rpn-popcorn__option">B) true — because '+' has a higher number (4 > 3), meaning it yields to '*'</button>
            <button onclick="checkPH1(2)" class="rpn-popcorn__option">C) false — because '+' has higher precedence than '*'</button>
            <button onclick="checkPH1(3)" class="rpn-popcorn__option">D) Compilation error — you can't compare Token objects</button>
        </div>
        <div id="ph1-feedback" class="rpn-popcorn__feedback"></div>
    </div>
</div>

<script>
function checkPH1(sel) {
    const btns = document.querySelectorAll('#ph1-options .rpn-popcorn__option');
    const fb = document.getElementById('ph1-feedback');
    btns.forEach(b => b.className = 'rpn-popcorn__option');
    if (sel === 1) {
        btns[1].classList.add('rpn-popcorn__option--correct');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--correct';
        fb.innerHTML = '<strong style="color:#2ecc71;">Correct!</strong> <code>isPrecedent</code> checks <code>this.precedence > other.getPrecedence()</code>. Since 4 > 3, it returns <strong>true</strong> — meaning <code>+</code> yields to <code>*</code> during the Shunting-Yard algorithm. Higher number = lower priority = yields.';
    } else {
        btns[sel].classList.add('rpn-popcorn__option--incorrect');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--incorrect';
        if (sel === 0) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> Precedence has nothing to do with alphabetical order — it\'s determined by the integer value passed to the constructor.';
        if (sel === 2) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> Actually, <code>+</code> has a <em>higher number</em> (4), which means <em>lower</em> priority. In math, <code>*</code> is evaluated before <code>+</code>.';
        if (sel === 3) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> <code>isPrecedent</code> is a regular method that compares integer values — no compilation error.';
    }
}
</script>


---

## Step 2: TermOrOperator — Subclass of Token

A `TermOrOperator` extends `Token` to also represent **numeric values**. This is an example of **inheritance** — the key AP CSA concept.

- If it's a **number**: stores the value as a String (e.g. `"100"`, `"3.14"`)
- If it's an **operator**: inherits the token, precedence, and calculation from Token
- If it's a **parenthesis**: just stores the character

**Why inheritance?** Both numbers and operators need to live in the same ArrayList. By making them share a parent class, we can store `ArrayList<TermOrOperator>` and treat them polymorphically.

### Challenge: Run the code to see how different TermOrOperator types behave.

In [ ]:
import java.util.function.BiFunction;

// --- Token (parent class) ---
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return this.calculation.apply(a, b); }
    public String toString() { return this.token.toString(); }
}

// --- TermOrOperator (child class) ---
class TermOrOperator extends Token {
    private final String value;

    // Constructor for numeric values
    public TermOrOperator(String value) {
        this.value = value;
    }

    // Constructor for parenthesis
    public TermOrOperator(Character token) {
        super(token);
        this.value = null;
    }

    // Constructor for binary operators
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation) {
        super(token, precedence, calculation, 2);
        this.value = null;
    }

    // Constructor for unary operators (like sqrt)
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        super(token, precedence, calculation, numArgs);
        this.value = null;
    }

    public String getValue() { return value; }

    public String toString() {
        return (this.value != null) ? this.value : super.toString();
    }
}

// --- TEST ---
TermOrOperator num = new TermOrOperator("42");
TermOrOperator plus = new TermOrOperator('+', 4, (a, b) -> a + b);
TermOrOperator openParen = new TermOrOperator('(');

System.out.println("Number token: " + num + " (value=" + num.getValue() + ")");
System.out.println("Operator token: " + plus + " (precedence=" + plus.getPrecedence() + ")");
System.out.println("Parenthesis token: " + openParen);
System.out.println();
System.out.println("Polymorphism demo — all in one ArrayList:");
java.util.ArrayList<TermOrOperator> list = new java.util.ArrayList<>();
list.add(openParen);
list.add(num);
list.add(plus);
list.add(new TermOrOperator("8"));
list.add(new TermOrOperator(')'));
System.out.println(list);

// TODO: Create a modulo operator (%, precedence 3) and test 10 % 3


{% capture challenge_termop %}
TermOrOperator extends Token to also hold numeric values. Run the code to see the three types: value, operator, and parenthesis. Then try adding a new operator for modulo (%).
{% endcapture %}

{% capture code_termop %}
import java.util.function.BiFunction;

// --- Token (parent class) ---
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return this.calculation.apply(a, b); }
    public String toString() { return this.token.toString(); }
}

// --- TermOrOperator (child class) ---
class TermOrOperator extends Token {
    private final String value;

    // Constructor for numeric values
    public TermOrOperator(String value) {
        this.value = value;
    }

    // Constructor for parenthesis
    public TermOrOperator(Character token) {
        super(token);
        this.value = null;
    }

    // Constructor for binary operators
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation) {
        super(token, precedence, calculation, 2);
        this.value = null;
    }

    // Constructor for unary operators (like sqrt)
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        super(token, precedence, calculation, numArgs);
        this.value = null;
    }

    public String getValue() { return value; }

    public String toString() {
        return (this.value != null) ? this.value : super.toString();
    }
}

// --- TEST ---
TermOrOperator num = new TermOrOperator("42");
TermOrOperator plus = new TermOrOperator('+', 4, (a, b) -> a + b);
TermOrOperator openParen = new TermOrOperator('(');

System.out.println("Number token: " + num + " (value=" + num.getValue() + ")");
System.out.println("Operator token: " + plus + " (precedence=" + plus.getPrecedence() + ")");
System.out.println("Parenthesis token: " + openParen);
System.out.println();
System.out.println("Polymorphism demo — all in one ArrayList:");
java.util.ArrayList<TermOrOperator> list = new java.util.ArrayList<>();
list.add(openParen);
list.add(num);
list.add(plus);
list.add(new TermOrOperator("8"));
list.add(new TermOrOperator(')'));
System.out.println(list);

// TODO: Create a modulo operator (%, precedence 3) and test 10 % 3
{% endcapture %}

{% include runners/code.html
   runner_id="rpn_termop"
   language="java"
   challenge=challenge_termop
   code=code_termop
   height="550px"
%}


<!-- MCQ_RUNNER: Numbers and Operators | id: rpn-polymorphism -->
```json
[
  {
    "q": "We store both numbers and operators in ArrayList<TermOrOperator>. Which AP CSA concept makes this possible?",
    "choices": [
      "Encapsulation — private fields hide implementation",
      "Polymorphism — subclass objects stored as parent type",
      "Abstraction — Token is an abstract class",
      "Overloading — multiple constructors in Token"
    ],
    "answer": 1,
    "ref": "RPN — Inheritance & Polymorphism",
    "why": "Polymorphism lets us declare an ArrayList<TermOrOperator> and store different subtypes in the same collection. When we call toString(), Java dynamically dispatches to the correct implementation: numbers print their value, while operators print their character."
  }
]
```


---

## Step 3: Tokens Map — Fast Operator Lookup

The `Tokens` class uses a **HashMap** to store all operators and separators. This gives us **O(1) lookup** — when we encounter a character in the expression, we can instantly check:

1. Is it an operator? → `operators.contains('+')`
2. What's its precedence? → `operators.getPrecedence('+')`
3. How does it calculate? → `operators.get('+').calculate(3, 5)`

**AP CSA Connection:** This is a real-world use of `HashMap<K, V>` — one of the most important data structures.

### Challenge: Run the code and then add the `^` (power) operator to the map.

In [ ]:
import java.util.Map;
import java.util.HashMap;
import java.util.function.BiFunction;

// Compact Token + TermOrOperator for this runner
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;
    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calc, int numArgs) {
        this.token = token; this.precedence = precedence; this.calculation = calc; this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calculation.apply(a, b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String value) { this.value = value; }
    public TermOrOperator(Character token) { super(token); this.value = null; }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        super(token, prec, calc, 2); this.value = null;
    }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        super(token, prec, calc, n); this.value = null;
    }
    public String getValue() { return value; }
    public String toString() { return value != null ? value : super.toString(); }
}

// --- Tokens Map (HashMap wrapper) ---
class Tokens {
    private Map<Character, TermOrOperator> map = new HashMap<>();

    public void put(Character token) {
        map.put(token, new TermOrOperator(token));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        map.put(token, new TermOrOperator(token, prec, calc));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        map.put(token, new TermOrOperator(token, prec, calc, n));
    }
    public TermOrOperator get(Character token) { return map.get(token); }
    public boolean contains(Character token) { return map.containsKey(token); }
    public String toString() { return map.toString(); }
}

// --- TEST ---
Tokens ops = new Tokens();
ops.put('+', 4, (a, b) -> a + b);
ops.put('-', 4, (a, b) -> a - b);
ops.put('*', 3, (a, b) -> a * b);
ops.put('/', 3, (a, b) -> a / b);
ops.put('%', 3, (a, b) -> a % b);

System.out.println("Operator map: " + ops);
System.out.println("Contains '+'? " + ops.contains('+'));
System.out.println("Contains 'x'? " + ops.contains('x'));
System.out.println("Precedence of '+': " + ops.get('+').getPrecedence());
System.out.println("Precedence of '*': " + ops.get('*').getPrecedence());
System.out.println("10 / 3 = " + ops.get('/').calculate(10.0, 3.0));
System.out.println("10 % 3 = " + ops.get('%').calculate(10.0, 3.0));

// TODO: Add power operator (^, precedence 2, Math.pow) and test 2^8


{% capture challenge_tokens %}
The Tokens class uses a HashMap for O(1) operator lookup. Run to see all registered operators. Then add the power operator (^, precedence 2, Math.pow).
{% endcapture %}

{% capture code_tokens %}
import java.util.Map;
import java.util.HashMap;
import java.util.function.BiFunction;

// Compact Token + TermOrOperator for this runner
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;
    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calc, int numArgs) {
        this.token = token; this.precedence = precedence; this.calculation = calc; this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calculation.apply(a, b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String value) { this.value = value; }
    public TermOrOperator(Character token) { super(token); this.value = null; }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        super(token, prec, calc, 2); this.value = null;
    }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        super(token, prec, calc, n); this.value = null;
    }
    public String getValue() { return value; }
    public String toString() { return value != null ? value : super.toString(); }
}

// --- Tokens Map (HashMap wrapper) ---
class Tokens {
    private Map<Character, TermOrOperator> map = new HashMap<>();

    public void put(Character token) {
        map.put(token, new TermOrOperator(token));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        map.put(token, new TermOrOperator(token, prec, calc));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        map.put(token, new TermOrOperator(token, prec, calc, n));
    }
    public TermOrOperator get(Character token) { return map.get(token); }
    public boolean contains(Character token) { return map.containsKey(token); }
    public String toString() { return map.toString(); }
}

// --- TEST ---
Tokens ops = new Tokens();
ops.put('+', 4, (a, b) -> a + b);
ops.put('-', 4, (a, b) -> a - b);
ops.put('*', 3, (a, b) -> a * b);
ops.put('/', 3, (a, b) -> a / b);
ops.put('%', 3, (a, b) -> a % b);

System.out.println("Operator map: " + ops);
System.out.println("Contains '+'? " + ops.contains('+'));
System.out.println("Contains 'x'? " + ops.contains('x'));
System.out.println("Precedence of '+': " + ops.get('+').getPrecedence());
System.out.println("Precedence of '*': " + ops.get('*').getPrecedence());
System.out.println("10 / 3 = " + ops.get('/').calculate(10.0, 3.0));
System.out.println("10 % 3 = " + ops.get('%').calculate(10.0, 3.0));

// TODO: Add power operator (^, precedence 2, Math.pow) and test 2^8
{% endcapture %}

{% include runners/code.html
   runner_id="rpn_tokens"
   language="java"
   challenge=challenge_tokens
   code=code_tokens
   height="550px"
%}


<!-- MCQ_RUNNER: Hashmap lookup time complexity | id: rpn-hashmap-lookup -->
```json
[
  {
    "q": "The Tokens class uses HashMap, TermOrOperator>. What is the time complexity of checking whether operators contains the key '+'?",
    "choices": [
      "O(n) — it scans through all entries",
      "O(log n) — it uses binary search",
      "O(1) — HashMap uses hashing for average-case constant-time lookup",
      "O(n²) — it compares every pair"
    ],
    "answer": 2,
    "ref": "RPN — HashMap Lookup",
    "why": "HashMap.containsKey() has O(1) average-case time complexity. It hashes the key character to locate its bucket. This makes HashMap a good choice over ArrayList for operator lookup when tokenizing expressions. Worst-case lookup can be O(n), but average-case performance is O(1)."
  }
]
```


---

## Step 4: Tokenizing an Expression

Before we can do any math, we need to **break the expression string into tokens**. This is called **tokenization** — the same concept compilers use to parse source code.

```
"(100 + 200) * 3" → [(, 100, +, 200, ), *, 3]
```

The algorithm walks character by character:
- If it's a **digit or decimal point** → accumulate into a multi-character number
- If it's an **operator or separator** → flush the accumulated number, then add the operator
- **Spaces** are separators but don't get added as tokens

### Challenge: Run the tokenizer on different expressions to see how it breaks them down.

In [ ]:
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double, Double, Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t, int p, BiFunction<Double,Double,Double> c, int n) { token=t; precedence=p; calc=c; numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c, int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}
class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c, int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

// --- Tokenizer ---
Tokens operators = new Tokens();
operators.put('+', 4, (a,b) -> a+b);
operators.put('-', 4, (a,b) -> a-b);
operators.put('*', 3, (a,b) -> a*b);
operators.put('/', 3, (a,b) -> a/b);
operators.put('%', 3, (a,b) -> a%b);
operators.put('^', 2, (a,b) -> Math.pow(a,b));

Tokens separators = new Tokens();
separators.put(' ');
separators.put('(');
separators.put(')');

ArrayList<TermOrOperator> tokenize(String expr, Tokens ops, Tokens seps) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start = 0;
    StringBuilder multi = new StringBuilder();
    for (int i = 0; i < expr.length(); i++) {
        char c = expr.charAt(i);
        if (ops.contains(c) || seps.contains(c)) {
            if (multi.length() > 0) {
                terms.add(new TermOrOperator(expr.substring(start, i)));
            }
            TermOrOperator t = ops.get(c);
            if (t == null) t = seps.get(c);
            if (t != null && t.getToken() != ' ') terms.add(t);
            start = i + 1;
            multi = new StringBuilder();
        } else {
            multi.append(c);
        }
    }
    if (multi.length() > 0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3.14 * 2",
    "2 ^ 10",
    "100 % 7"
};
for (String expr : expressions) {
    System.out.println("\"" + expr + "\"  →  " + tokenize(expr, operators, separators));
}

// TODO: Try tokenizing your own expression!


{% capture challenge_tokenize %}
The tokenizer splits a math expression string into individual tokens. Run it to see how different expressions get tokenized. Try adding your own expression at the bottom.
{% endcapture %}

{% capture code_tokenize %}
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double, Double, Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t, int p, BiFunction<Double,Double,Double> c, int n) { token=t; precedence=p; calc=c; numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c, int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}
class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c, int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

// --- Tokenizer ---
Tokens operators = new Tokens();
operators.put('+', 4, (a,b) -> a+b);
operators.put('-', 4, (a,b) -> a-b);
operators.put('*', 3, (a,b) -> a*b);
operators.put('/', 3, (a,b) -> a/b);
operators.put('%', 3, (a,b) -> a%b);
operators.put('^', 2, (a,b) -> Math.pow(a,b));

Tokens separators = new Tokens();
separators.put(' ');
separators.put('(');
separators.put(')');

ArrayList<TermOrOperator> tokenize(String expr, Tokens ops, Tokens seps) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start = 0;
    StringBuilder multi = new StringBuilder();
    for (int i = 0; i < expr.length(); i++) {
        char c = expr.charAt(i);
        if (ops.contains(c) || seps.contains(c)) {
            if (multi.length() > 0) {
                terms.add(new TermOrOperator(expr.substring(start, i)));
            }
            TermOrOperator t = ops.get(c);
            if (t == null) t = seps.get(c);
            if (t != null && t.getToken() != ' ') terms.add(t);
            start = i + 1;
            multi = new StringBuilder();
        } else {
            multi.append(c);
        }
    }
    if (multi.length() > 0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3.14 * 2",
    "2 ^ 10",
    "100 % 7"
};
for (String expr : expressions) {
    System.out.println("\"" + expr + "\"  →  " + tokenize(expr, operators, separators));
}

// TODO: Try tokenizing your own expression!
{% endcapture %}

{% include runners/code.html
   runner_id="rpn_tokenize"
   language="java"
   challenge=challenge_tokenize
   code=code_tokenize
   height="550px"
%}



<!-- MCQ_RUNNER: Tokenization | id: rpn-tokenization-trace -->
```json
[
  {
    "q": "Given the expression <code>\"(100 + 200) * 3\"</code>, what token list should the tokenizer produce?",
    "choices": [
      "[\"(\", \"100\", \"+\", \"200\", \")\", \"*\", \"3\"]",
      "[\"(\", \"1\", \"0\", \"0\", \"+\", \"2\", \"0\", \"0\", \")\", \"*\", \"3\"]",
      "[\"(\", \"100\", \" \", \"+\", \" \", \"200\", \")\", \" \", \"*\", \" \", \"3\"]",
      "[\"100\", \"+\", \"200\", \"*\", \"3\"]"
    ],
    "answer": 0,
    "ref": "RPN — Tokenization",
    "why": "The tokenizer accumulates consecutive digits into multi-character number tokens, adds operators and parentheses as individual tokens, and uses spaces as separators without adding them to the result. The final list contains seven tokens: (, 100, +, 200, ), *, 3."
  }
]
```


---

## Step 5: Converting to Reverse Polish Notation (The Stack Algorithm)

This is the **core algorithm**. RPN reorders tokens so that operators come AFTER their operands, eliminating the need for parentheses.

**Infix (normal):** `(100 + 200) * 3`  
**RPN (postfix):** `100 200 + 3 *`

### The Shunting-Yard Algorithm (by Dijkstra)

Uses a **Stack** to reorder operators:

1. **Number** → goes straight to output
2. **`(`** → push onto operator stack
3. **`)`** → pop operators to output until `(` is found
4. **Operator** → pop higher-precedence operators from stack to output, then push current
5. **End** → pop remaining operators to output

### Challenge: Trace the algorithm by hand for `(100 + 200) * 3`, then run to verify.

In [ ]:
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) { token=t;precedence=p;calc=c;numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence>t.getPrecedence(); }
    public Double calculate(Double a,Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null?value:super.toString(); }
}
class Tokens {
    Map<Character,TermOrOperator> map=new HashMap<>();
    void put(Character t) { map.put(t,new TermOrOperator(t)); }
    void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t,new TermOrOperator(t,p,c)); }
    TermOrOperator get(Character t) { return map.get(t); }
    boolean contains(Character t) { return map.containsKey(t); }
}

Tokens ops = new Tokens();
ops.put('+',4,(a,b)->a+b); ops.put('-',4,(a,b)->a-b);
ops.put('*',3,(a,b)->a*b); ops.put('/',3,(a,b)->a/b);
ops.put('^',2,(a,b)->Math.pow(a,b));
Tokens seps = new Tokens();
seps.put(' '); seps.put('('); seps.put(')');

ArrayList<TermOrOperator> tokenize(String expr) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start=0; StringBuilder multi=new StringBuilder();
    for (int i=0;i<expr.length();i++) {
        char c=expr.charAt(i);
        if (ops.contains(c)||seps.contains(c)) {
            if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start,i)));
            TermOrOperator t=ops.get(c); if(t==null) t=seps.get(c);
            if (t!=null && t.getToken()!=' ') terms.add(t);
            start=i+1; multi=new StringBuilder();
        } else multi.append(c);
    }
    if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- Shunting-Yard: Infix → RPN ---
ArrayList<TermOrOperator> toRPN(ArrayList<TermOrOperator> terms) {
    ArrayList<TermOrOperator> rpn = new ArrayList<>();
    Stack<TermOrOperator> opStack = new Stack<>();
    for (TermOrOperator term : terms) {
        if (term.getToken() == '(') {
            opStack.push(term);
        } else if (term.getToken() == ')') {
            while (opStack.peek().getToken() != '(') rpn.add(opStack.pop());
            opStack.pop(); // remove '('
        } else if (ops.contains(term.getToken())) {
            while (!opStack.isEmpty() && ops.contains(opStack.peek().getToken()) && term.isPrecedent(opStack.peek()))
                rpn.add(opStack.pop());
            opStack.push(term);
        } else {
            rpn.add(term);
        }
    }
    while (!opStack.isEmpty()) rpn.add(opStack.pop());
    return rpn;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3 + 4 * 2 - 1",
    "2 ^ 10"
};
for (String expr : expressions) {
    ArrayList<TermOrOperator> tokens = tokenize(expr);
    ArrayList<TermOrOperator> rpn = toRPN(tokens);
    System.out.println("Infix:  " + expr);
    System.out.println("Tokens: " + tokens);
    System.out.println("RPN:    " + rpn);
    System.out.println();
}


{% capture challenge_rpn %}
The Shunting-Yard algorithm converts infix to RPN using a Stack. Run to see step-by-step conversion. Try tracing "5 + 3 * 2" by hand first, then verify with the code.
{% endcapture %}

{% capture code_rpn %}
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) { token=t;precedence=p;calc=c;numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence>t.getPrecedence(); }
    public Double calculate(Double a,Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null?value:super.toString(); }
}
class Tokens {
    Map<Character,TermOrOperator> map=new HashMap<>();
    void put(Character t) { map.put(t,new TermOrOperator(t)); }
    void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t,new TermOrOperator(t,p,c)); }
    TermOrOperator get(Character t) { return map.get(t); }
    boolean contains(Character t) { return map.containsKey(t); }
}

Tokens ops = new Tokens();
ops.put('+',4,(a,b)->a+b); ops.put('-',4,(a,b)->a-b);
ops.put('*',3,(a,b)->a*b); ops.put('/',3,(a,b)->a/b);
ops.put('^',2,(a,b)->Math.pow(a,b));
Tokens seps = new Tokens();
seps.put(' '); seps.put('('); seps.put(')');

ArrayList<TermOrOperator> tokenize(String expr) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start=0; StringBuilder multi=new StringBuilder();
    for (int i=0;i<expr.length();i++) {
        char c=expr.charAt(i);
        if (ops.contains(c)||seps.contains(c)) {
            if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start,i)));
            TermOrOperator t=ops.get(c); if(t==null) t=seps.get(c);
            if (t!=null && t.getToken()!=' ') terms.add(t);
            start=i+1; multi=new StringBuilder();
        } else multi.append(c);
    }
    if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- Shunting-Yard: Infix to RPN ---
ArrayList<TermOrOperator> toRPN(ArrayList<TermOrOperator> terms) {
    ArrayList<TermOrOperator> rpn = new ArrayList<>();
    Stack<TermOrOperator> opStack = new Stack<>();
    for (TermOrOperator term : terms) {
        if (term.getToken() == '(') {
            opStack.push(term);
        } else if (term.getToken() == ')') {
            while (opStack.peek().getToken() != '(') rpn.add(opStack.pop());
            opStack.pop(); // remove '('
        } else if (ops.contains(term.getToken())) {
            while (!opStack.isEmpty() && ops.contains(opStack.peek().getToken()) && term.isPrecedent(opStack.peek()))
                rpn.add(opStack.pop());
            opStack.push(term);
        } else {
            rpn.add(term);
        }
    }
    while (!opStack.isEmpty()) rpn.add(opStack.pop());
    return rpn;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3 + 4 * 2 - 1",
    "2 ^ 10"
};
for (String expr : expressions) {
    ArrayList<TermOrOperator> tokens = tokenize(expr);
    ArrayList<TermOrOperator> rpn = toRPN(tokens);
    System.out.println("Infix:  " + expr);
    System.out.println("Tokens: " + tokens);
    System.out.println("RPN:    " + rpn);
    System.out.println();
}
{% endcapture %}

{% include runners/code.html
   runner_id="rpn_convert"
   language="java"
   challenge=challenge_rpn
   code=code_rpn
   height="550px"
%}


<div class="ocs__dnd ocs__dnd--medium" id="rpn-ordering">
  <div class="ocs__dnd-header">
    <h3>RPN Ordering Challenge</h3>
    <p>Drag the tokens into the correct Reverse Polish Notation order for: <strong>(5 + 3) * 2</strong></p>
    <div class="ocs__dnd-status" aria-live="polite">Arrange all five tokens.</div>
    <div class="ocs__dnd-progress"></div>
  </div>

  <div class="ocs__dnd-layout">
    <div class="ocs__dnd-panel" id="rpn-source">
      <h4 class="ocs__dnd-panel-title">Available Tokens</h4>
      <div id="rpn-available"></div>
    </div>

    <div class="ocs__dnd-panel" id="rpn-target">
      <h4 class="ocs__dnd-panel-title">Your RPN Order</h4>
      <div id="rpn-positions"></div>
    </div>
  </div>

  <div class="ocs__dnd-controls">
    <button type="button" class="ocs__dnd-reset" data-action="check">Check Answer</button>
    <button type="button" class="ocs__dnd-reset" data-action="clear">Clear All</button>
    <button type="button" class="ocs__dnd-reset" data-action="reset">Reset &amp; Shuffle</button>
  </div>
</div>

<script>
(function () {
  const board = document.getElementById('rpn-ordering');
  if (!board || board.dataset.initialized) return;
  board.dataset.initialized = 'true';

  const correctOrder = ['5', '3', '+', '2', '*'];
  const originalTokens = ['*', '5', '+', '3', '2'];

  const available = board.querySelector('#rpn-available');
  const positions = board.querySelector('#rpn-positions');
  const status = board.querySelector('.ocs__dnd-status');
  const progress = board.querySelector('.ocs__dnd-progress');

  let tokens = [];
  let placed = Array(5).fill(null);
  let selected = null;
  let dragged = null;

  function shuffle(items) {
    const result = [...items];
    for (let i = result.length - 1; i > 0; i--) {
      const j = Math.floor(Math.random() * (i + 1));
      [result[i], result[j]] = [result[j], result[i]];
    }
    return result;
  }

  function updateStatus(message) {
    status.textContent = message;
  }

  function updateProgress() {
    const count = placed.filter(Boolean).length;
    progress.textContent = count + ' of 5 positions filled';
  }

  function render() {
    available.innerHTML = '';
    positions.innerHTML = '';

    tokens.forEach(token => {
      if (placed.includes(token)) return;

      const button = document.createElement('button');
      button.type = 'button';
      button.className = 'ocs__drag-btn';
      button.textContent = token;
      button.draggable = true;
      button.dataset.token = token;

      if (selected === token) button.classList.add('is-selected');

      button.addEventListener('click', () => {
        selected = selected === token ? null : token;
        render();
      });

      button.addEventListener('dragstart', event => {
        dragged = token;
        event.dataTransfer.setData('text/plain', token);
        event.dataTransfer.effectAllowed = 'move';
      });

      button.addEventListener('dragend', () => {
        dragged = null;
      });

      available.appendChild(button);
    });

    placed.forEach((token, index) => {
      const button = document.createElement('button');
      button.type = 'button';
      button.className = 'ocs__drop-btn';
      button.dataset.position = index;

      if (token) {
        button.textContent = token;
        button.classList.add('is-filled');
        button.draggable = true;

        button.addEventListener('dragstart', event => {
          dragged = token;
          event.dataTransfer.setData('text/plain', token);
          event.dataTransfer.effectAllowed = 'move';
        });
      } else {
        button.textContent = 'Position ' + (index + 1) + ' — Drop here';
      }

      button.addEventListener('click', () => {
        if (selected) {
          placeToken(selected, index);
        } else if (placed[index]) {
          selected = placed[index];
          placed[index] = null;
          render();
        }
      });

      button.addEventListener('dragover', event => {
        event.preventDefault();
        button.classList.add('is-over');
      });

      button.addEventListener('dragleave', () => {
        button.classList.remove('is-over');
      });

      button.addEventListener('drop', event => {
        event.preventDefault();
        button.classList.remove('is-over');

        const tokenToPlace = event.dataTransfer.getData('text/plain') || dragged;
        if (tokenToPlace) placeToken(tokenToPlace, index);
      });

      positions.appendChild(button);
    });

    updateProgress();
  }

  function placeToken(token, index) {
    if (!tokens.includes(token)) return;

    const previousIndex = placed.indexOf(token);
    if (previousIndex !== -1) placed[previousIndex] = null;

    placed[index] = token;
    selected = null;

    updateStatus('Token placed. Continue arranging the sequence.');
    render();
  }

  function checkAnswer() {
    if (placed.some(token => token === null)) {
      updateStatus('Incomplete! Fill all five positions before checking.');
      return;
    }

    const correct = placed.every((token, index) =>
      token === correctOrder[index]
    );

    if (correct) {
      updateStatus(
        'Perfect! 5 3 + 2 * is correct RPN for (5 + 3) * 2. ' +
        'Push 5, push 3, add to get 8, push 2, then multiply to get 16.'
      );
    } else {
      updateStatus(
        'Not quite. In RPN, operands come before their operator. ' +
        'The expression (5 + 3) means 5 and 3 must appear before +.'
      );
    }
  }

  function clearBoard() {
    placed = Array(5).fill(null);
    selected = null;
    updateStatus('Board cleared. Arrange the tokens again.');
    render();
  }

  function resetBoard() {
    tokens = shuffle(originalTokens);
    placed = Array(5).fill(null);
    selected = null;
    updateStatus('Tokens shuffled. Arrange all five positions.');
    render();
  }

  board.querySelector('[data-action="check"]').addEventListener('click', checkAnswer);
  board.querySelector('[data-action="clear"]').addEventListener('click', clearBoard);
  board.querySelector('[data-action="reset"]').addEventListener('click', resetBoard);

  resetBoard();
})();
</script>


---

## Step 6: Evaluating RPN — The Final Calculation

Once we have RPN, evaluation is beautifully simple using a **Stack**:

1. Read tokens left to right
2. **Number** → push onto stack
3. **Operator** → pop operand(s), calculate, push result back
4. **Done** → one number remains on stack = final answer

### Example: `100 200 + 3 *`

| Step | Token | Stack | Action |
|------|-------|-------|--------|
| 1 | 100 | [100] | Push |
| 2 | 200 | [100, 200] | Push |
| 3 | + | [300] | Pop 200 & 100, add, push 300 |
| 4 | 3 | [300, 3] | Push |
| 5 | * | [900] | Pop 3 & 300, multiply, push 900 |

**Result: 900**


<div class="rpn-stack-viz">
    <div class="rpn-stack-viz__header">
        <h3>Interactive Stack Visualization</h3>
        <p>Step through RPN evaluation of <code>100 200 + 3 *</code> and watch the stack in action.</p>
    </div>
    <div class="rpn-stack-viz__body">
        <div class="rpn-stack-viz__controls">
            <button class="rpn-stack-viz__step-btn" id="stackStepBtn" onclick="stackStep()">Next Step</button>
            <button class="rpn-stack-viz__step-btn rpn-stack-viz__step-btn--reset" onclick="stackReset()">Reset</button>
        </div>
        <div class="rpn-stack-viz__display">
            <div class="rpn-stack-viz__token-stream">
                <h5>Token Stream</h5>
                <div id="stackTokens"></div>
            </div>
            <div class="rpn-stack-viz__stack-container">
                <h5>Stack</h5>
                <div class="rpn-stack-viz__stack" id="stackDisplay"></div>
            </div>
        </div>
        <div class="rpn-stack-viz__narration" id="stackNarration">Click <strong>Next Step</strong> to begin evaluation.</div>
    </div>
</div>

<script>
(function() {
    const rpnTokens = ['100', '200', '+', '3', '*'];
    const steps = [
        { active: 0, stack: ['100'], narration: '<strong>Push 100</strong> — it\'s a number, so push it onto the stack.' },
        { active: 1, stack: ['100', '200'], narration: '<strong>Push 200</strong> — another number, push it.' },
        { active: 2, stack: ['300'], narration: '<strong>Operator +</strong> — pop 200 and 100, calculate 100 + 200 = 300, push result.' },
        { active: 3, stack: ['300', '3'], narration: '<strong>Push 3</strong> — number goes on the stack.' },
        { active: 4, stack: ['900'], narration: '<strong>Operator *</strong> — pop 3 and 300, calculate 300 x 3 = 900, push result.' },
        { active: -1, stack: ['900'], narration: '<strong>Done!</strong> One value remains on the stack: <strong>900</strong> — that\'s our answer!' }
    ];
    let currentStep = -1;

    function renderTokenStream() {
        const container = document.getElementById('stackTokens');
        container.innerHTML = '';
        rpnTokens.forEach((t, i) => {
            const el = document.createElement('span');
            el.className = 'rpn-stack-viz__token';
            if (currentStep >= 0 && i < steps[currentStep].active) el.classList.add('rpn-stack-viz__token--processed');
            if (currentStep >= 0 && i === steps[currentStep].active) el.classList.add('rpn-stack-viz__token--active');
            el.textContent = t;
            container.appendChild(el);
        });
    }

    function renderStack() {
        const container = document.getElementById('stackDisplay');
        container.innerHTML = '';
        if (currentStep >= 0) {
            steps[currentStep].stack.forEach(val => {
                const el = document.createElement('div');
                el.className = 'rpn-stack-viz__stack-item';
                el.textContent = val;
                container.appendChild(el);
            });
        }
    }

    function updateNarration() {
        const el = document.getElementById('stackNarration');
        if (currentStep < 0) el.innerHTML = 'Click <strong>Next Step</strong> to begin evaluation.';
        else el.innerHTML = steps[currentStep].narration;
    }

    window.stackStep = function() {
        if (currentStep < steps.length - 1) {
            currentStep++;
            renderTokenStream();
            renderStack();
            updateNarration();
        }
        if (currentStep >= steps.length - 1) {
            document.getElementById('stackStepBtn').disabled = true;
        }
    };

    window.stackReset = function() {
        currentStep = -1;
        document.getElementById('stackStepBtn').disabled = false;
        renderTokenStream();
        renderStack();
        updateNarration();
    };

    renderTokenStream();
    renderStack();
    updateNarration();
})();
</script>


---

## Step 7: The Complete Calculator

Now let's put it all together. The full `Calculator` class chains the entire pipeline:

```
Expression → Tokenize → RPN → Evaluate → Result
```

### Challenge: Run the complete calculator, then try your own expressions!

Supported operators: `+`, `-`, `*`, `/`, `%`, `^` (power), `√` (square root)

In [ ]:
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) {
        token=t; precedence=p; calc=c; numArgs=n;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}

class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c,int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public int getPrecedence(Character t) { return map.get(t).getPrecedence(); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

public class Calculator {
    private final String expression;
    private ArrayList<TermOrOperator> terms = new ArrayList<>();
    private ArrayList<TermOrOperator> rpnTerms = new ArrayList<>();
    private Tokens operators = new Tokens();
    private Tokens separators = new Tokens();
    private Double result = 0.0;

    public Calculator(String expression) {
        initOperators();
        initSeparators();
        this.expression = expression;
        this.termTokenizer();
        this.termsToRPN();
        this.rpnToResult();
    }

    private void initOperators() {
        operators.put('*', 3, (a,b) -> a*b);
        operators.put('/', 3, (a,b) -> a/b);
        operators.put('%', 3, (a,b) -> a%b);
        operators.put('+', 4, (a,b) -> a+b);
        operators.put('-', 4, (a,b) -> a-b);
        operators.put('^', 2, (a,b) -> Math.pow(a,b));
    }

    private void initSeparators() {
        separators.put(' ');
        separators.put('(');
        separators.put(')');
    }

    private void termTokenizer() {
        int start = 0;
        StringBuilder multi = new StringBuilder();
        for (int i = 0; i < expression.length(); i++) {
            char c = expression.charAt(i);
            if (operators.contains(c) || separators.contains(c)) {
                if (multi.length() > 0)
                    terms.add(new TermOrOperator(expression.substring(start, i)));
                TermOrOperator t = operators.get(c);
                if (t == null) t = separators.get(c);
                if (t != null && t.getToken() != ' ') terms.add(t);
                start = i + 1;
                multi = new StringBuilder();
            } else {
                multi.append(c);
            }
        }
        if (multi.length() > 0) terms.add(new TermOrOperator(expression.substring(start)));
    }

    private void termsToRPN() {
        Stack<TermOrOperator> opStack = new Stack<>();
        for (TermOrOperator term : terms) {
            if (term.getToken() == '(') {
                opStack.push(term);
            } else if (term.getToken() == ')') {
                while (opStack.peek().getToken() != '(') rpnTerms.add(opStack.pop());
                opStack.pop();
            } else if (operators.contains(term.getToken())) {
                while (!opStack.isEmpty() && operators.contains(opStack.peek().getToken())
                       && term.isPrecedent(opStack.peek()))
                    rpnTerms.add(opStack.pop());
                opStack.push(term);
            } else {
                rpnTerms.add(term);
            }
        }
        while (!opStack.isEmpty()) rpnTerms.add(opStack.pop());
    }

    private void rpnToResult() {
        Stack<Double> calcStack = new Stack<>();
        for (TermOrOperator term : rpnTerms) {
            if (operators.contains(term.getToken())) {
                Double op2 = calcStack.pop();
                Double op1 = calcStack.pop();
                calcStack.push(term.calculate(op1, op2));
            } else {
                calcStack.push(Double.valueOf(term.getValue()));
            }
        }
        this.result = calcStack.pop();
    }

    public String toString() {
        return "Original expression: " + expression + "\n"
             + "Tokenized expression: " + terms + "\n"
             + "Reverse Polish Notation: " + rpnTerms + "\n"
             + "Final result: " + String.format("%.2f", result);
    }

    public static void main(String[] args) {
        System.out.println("=== Simple Math ===");
        System.out.println(new Calculator("100 + 200 * 3"));
        System.out.println();

        System.out.println("=== Parentheses Override Precedence ===");
        System.out.println(new Calculator("(100 + 200) * 3"));
        System.out.println();

        System.out.println("=== Decimals ===");
        System.out.println(new Calculator("100.2 - 99.3"));
        System.out.println();

        System.out.println("=== Modulo ===");
        System.out.println(new Calculator("300 % 200"));
        System.out.println();

        System.out.println("=== Power ===");
        System.out.println(new Calculator("2 ^ 10"));
        System.out.println();

        // TODO: Add your own expression here!
        // System.out.println(new Calculator("YOUR EXPRESSION"));
    }
}
Calculator.main(null);


{% capture challenge_full %}
The complete RPN Calculator! Run it to see the full pipeline — from expression to tokenization to RPN to result. Try adding your own expressions at the bottom of main().
{% endcapture %}

{% capture code_full %}
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) {
        token=t; precedence=p; calc=c; numArgs=n;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}

class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c,int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public int getPrecedence(Character t) { return map.get(t).getPrecedence(); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

public class Calculator {
    private final String expression;
    private ArrayList<TermOrOperator> terms = new ArrayList<>();
    private ArrayList<TermOrOperator> rpnTerms = new ArrayList<>();
    private Tokens operators = new Tokens();
    private Tokens separators = new Tokens();
    private Double result = 0.0;

    public Calculator(String expression) {
        initOperators();
        initSeparators();
        this.expression = expression;
        this.termTokenizer();
        this.termsToRPN();
        this.rpnToResult();
    }

    private void initOperators() {
        operators.put('*', 3, (a,b) -> a*b);
        operators.put('/', 3, (a,b) -> a/b);
        operators.put('%', 3, (a,b) -> a%b);
        operators.put('+', 4, (a,b) -> a+b);
        operators.put('-', 4, (a,b) -> a-b);
        operators.put('^', 2, (a,b) -> Math.pow(a,b));
    }

    private void initSeparators() {
        separators.put(' ');
        separators.put('(');
        separators.put(')');
    }

    private void termTokenizer() {
        int start = 0;
        StringBuilder multi = new StringBuilder();
        for (int i = 0; i < expression.length(); i++) {
            char c = expression.charAt(i);
            if (operators.contains(c) || separators.contains(c)) {
                if (multi.length() > 0)
                    terms.add(new TermOrOperator(expression.substring(start, i)));
                TermOrOperator t = operators.get(c);
                if (t == null) t = separators.get(c);
                if (t != null && t.getToken() != ' ') terms.add(t);
                start = i + 1;
                multi = new StringBuilder();
            } else {
                multi.append(c);
            }
        }
        if (multi.length() > 0) terms.add(new TermOrOperator(expression.substring(start)));
    }

    private void termsToRPN() {
        Stack<TermOrOperator> opStack = new Stack<>();
        for (TermOrOperator term : terms) {
            if (term.getToken() == '(') {
                opStack.push(term);
            } else if (term.getToken() == ')') {
                while (opStack.peek().getToken() != '(') rpnTerms.add(opStack.pop());
                opStack.pop();
            } else if (operators.contains(term.getToken())) {
                while (!opStack.isEmpty() && operators.contains(opStack.peek().getToken())
                       && term.isPrecedent(opStack.peek()))
                    rpnTerms.add(opStack.pop());
                opStack.push(term);
            } else {
                rpnTerms.add(term);
            }
        }
        while (!opStack.isEmpty()) rpnTerms.add(opStack.pop());
    }

    private void rpnToResult() {
        Stack<Double> calcStack = new Stack<>();
        for (TermOrOperator term : rpnTerms) {
            if (operators.contains(term.getToken())) {
                Double op2 = calcStack.pop();
                Double op1 = calcStack.pop();
                calcStack.push(term.calculate(op1, op2));
            } else {
                calcStack.push(Double.valueOf(term.getValue()));
            }
        }
        this.result = calcStack.pop();
    }

    public String toString() {
        return "Original expression: " + expression + "\n"
             + "Tokenized expression: " + terms + "\n"
             + "Reverse Polish Notation: " + rpnTerms + "\n"
             + "Final result: " + String.format("%.2f", result);
    }

    public static void main(String[] args) {
        System.out.println("=== Simple Math ===");
        System.out.println(new Calculator("100 + 200 * 3"));
        System.out.println();

        System.out.println("=== Parentheses Override Precedence ===");
        System.out.println(new Calculator("(100 + 200) * 3"));
        System.out.println();

        System.out.println("=== Decimals ===");
        System.out.println(new Calculator("100.2 - 99.3"));
        System.out.println();

        System.out.println("=== Modulo ===");
        System.out.println(new Calculator("300 % 200"));
        System.out.println();

        System.out.println("=== Power ===");
        System.out.println(new Calculator("2 ^ 10"));
        System.out.println();

        // TODO: Add your own expression here!
        // System.out.println(new Calculator("YOUR EXPRESSION"));
    }
}
Calculator.main(null);
{% endcapture %}

{% include runners/code.html
   runner_id="rpn_full"
   language="java"
   challenge=challenge_full
   code=code_full
   height="600px"
%}

## Knowledge Check


<!-- MCQ_RUNNER: MCQ Summary | id: rpn-final-knowledge-check -->
```json
[
  {
    "q": "What data structure does the Shunting-Yard algorithm use to temporarily hold operators during the infix-to-RPN conversion?",
    "choices": [
      "Queue",
      "Stack",
      "HashMap",
      "LinkedList"
    ],
    "answer": 1,
    "ref": "RPN Calculator — Shunting-Yard Algorithm",
    "why": "The Shunting-Yard algorithm uses a Stack (LIFO) to hold operators temporarily. When a higher-precedence operator is encountered, lower-precedence operators are popped from the stack to the output."
  },
  {
    "q": "What is the RPN (postfix) form of <code>3 + 4 * 2</code>?",
    "choices": [
      "3 4 + 2 *",
      "3 4 2 * +",
      "+ 3 * 4 2",
      "3 4 2 + *"
    ],
    "answer": 1,
    "ref": "RPN Calculator — Infix-to-RPN Conversion",
    "why": "<code>3 4 2 * +</code> is correct. Multiplication has higher precedence than addition, so <code>4 * 2</code> is evaluated first. Its RPN form is <code>4 2 *</code>, followed by the addition with 3."
  },
  {
    "q": "In the Token class, <code>BiFunction<Double, Double, Double></code> is used for the <code>calculation</code> field. What type of Java feature is BiFunction?",
    "choices": [
      "Abstract class",
      "Functional interface",
      "Enum",
      "Generic class"
    ],
    "answer": 1,
    "ref": "RPN Calculator — Functional Interfaces",
    "why": "<code>BiFunction</code> is a functional interface in <code>java.util.function</code>. It has one abstract method, <code>apply</code>, and supports lambda expressions such as <code>(a, b) -> a + b</code>."
  },
  {
    "q": "During RPN evaluation of <code>5 3 + 2 *</code>, what is on the stack after processing the <code>+</code> token?",
    "choices": [
      "[5, 3, +]",
      "[8]",
      "[5, 8]",
      "[3, 5]"
    ],
    "answer": 1,
    "ref": "RPN Calculator — Stack Evaluation",
    "why": "When the + operator is encountered, the evaluator pops 3 and 5, computes 5 + 3 = 8, and pushes the result. The stack now contains [8]."
  },
  {
    "q": "When the tokenizer encounters a <code>)</code> character, what does it do?",
    "choices": [
      "Ignores it — parentheses are not real tokens",
      "Flushes any accumulated number, then adds <code>)</code> as a separator token",
      "Pushes it directly onto the operator stack",
      "Ends the tokenization process"
    ],
    "answer": 1,
    "ref": "RPN Calculator — Tokenization",
    "why": "The tokenizer first flushes any accumulated multi-digit number, then adds <code>)</code> as a token to the token list. Parentheses are necessary for the Shunting-Yard algorithm to group operations correctly."
  }
]
```


---

## Key Takeaways

### Data Structures Used

| Data Structure | Where | Purpose |
|----------------|-------|---------|
| **ArrayList** | `terms`, `rpnTerms` | Ordered storage of tokens |
| **Stack** | Shunting-yard & evaluation | LIFO for operator reordering and calculation |
| **HashMap** | `Tokens` class | O(1) lookup of operators by character |

### AP CSA Concepts Demonstrated

- **Inheritance** — `TermOrOperator extends Token`
- **Polymorphism** — numbers and operators stored in the same `ArrayList<TermOrOperator>`
- **Encapsulation** — private fields with getters
- **Functional Interfaces** — `BiFunction<Double, Double, Double>` for operator calculations
- **Algorithmic Thinking** — Dijkstra's Shunting-Yard algorithm

### Hack Ideas

1. Add support for **unary minus** (negative numbers like `-5 + 3`)
2. Add **trigonometric functions** (`sin`, `cos`, `tan`)
3. Add **error handling** for malformed expressions (mismatched parentheses, division by zero)
4. Build a **visual step-by-step debugger** that shows the stack state at each step
5. Create a **GUI calculator** that uses this engine under the hood